# 👥 Module 4: RFM Customer Segmentation (KMeans Clustering)

In this notebook, we calculate **Recency, Frequency, and Monetary (RFM)** metrics for each customer, normalize the feature values, train a **KMeans Clustering** model, assign human-readable segment names (**VIP**, **Loyal**, **Regular**, **At Risk**), and export trained models.

---

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import joblib
import plotly.express as px

cleaned_path = os.path.join("..", "data", "processed", "cleaned_retail_data.csv")
df = pd.read_csv(cleaned_path)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
print(f"Loaded dataset: {df.shape[0]:,} rows")

Loaded dataset: 392,692 rows


### Step 1: Calculate RFM Metrics per Customer

In [2]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f"Snapshot Date for Recency: {snapshot_date}")

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']
rfm = rfm[rfm['Monetary'] > 0].reset_index(drop=True)
rfm.head()

Snapshot Date for Recency: 2011-12-10 12:50:00


,CustomerID,Recency,Frequency,Monetary
0,12346,326,1,77183.60
1,12347,2,7,4310.00
2,12348,75,4,1797.24
3,12349,19,1,1757.55
4,12350,310,1,334.40


### Step 2: Log Transformation & Standard Scaling

In [3]:
# Handle skewness using log1p
rfm_log = np.log1p(rfm[['Recency', 'Frequency', 'Monetary']])

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)
print(f"Scaled Feature Matrix Shape: {rfm_scaled.shape}")

Scaled Feature Matrix Shape: (4338, 3)


### Step 3: Train KMeans Model & Assign Clusters

In [4]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

# Calculate Cluster Means
cluster_means = rfm.groupby('Cluster').agg({'Recency':'mean', 'Frequency':'mean', 'Monetary':'mean'})
print("Cluster Averages:\n", cluster_means)

Cluster Averages:
             Recency  Frequency     Monetary
Cluster                                    
0         12.171108  13.751753  8088.018331
1        181.506782   1.316893   340.998626
2         17.698925   2.192354   557.322616
3         71.635506   4.078902  1801.777893


### Step 4: Map Clusters to Customer Segments

In [5]:
# Score clusters based on high Monetary + Frequency, low Recency
scores = (
    (cluster_means['Monetary'] / cluster_means['Monetary'].max()) +
    (cluster_means['Frequency'] / cluster_means['Frequency'].max()) -
    (cluster_means['Recency'] / cluster_means['Recency'].max())
)
sorted_clusters = scores.sort_values(ascending=False).index.tolist()

segment_names = ["VIP Customers", "Loyal Customers", "Regular Customers", "At Risk Customers"]
label_map = {cluster_id: segment_names[i] for i, cluster_id in enumerate(sorted_clusters)}

rfm['Segment'] = rfm['Cluster'].map(label_map)
rfm['Segment'].value_counts()

Segment
At Risk Customers    1622
Regular Customers    1166
Loyal Customers       837
VIP Customers         713
Name: count, dtype: int64

### Step 5: 3D Visualization of Customer Clusters

In [6]:
fig = px.scatter_3d(
    rfm, x='Recency', y='Frequency', z='Monetary',
    color='Segment', title='👥 3D Customer Segments (RFM Clusters)',
    color_discrete_map={
        "VIP Customers": "#10B981",
        "Loyal Customers": "#3B82F6",
        "Regular Customers": "#F59E0B",
        "At Risk Customers": "#EF4444"
    }
)
fig.show()

### Step 6: Export Trained Models & RFM Data

In [7]:
model_dir = os.path.join("..", "models")
os.makedirs(model_dir, exist_ok=True)

joblib.dump(kmeans, os.path.join(model_dir, "kmeans_rfm.joblib"))
joblib.dump(scaler, os.path.join(model_dir, "scaler_rfm.joblib"))
rfm.to_csv(os.path.join("..", "data", "processed", "rfm_data.csv"), index=False)
print("Trained segmentation models saved successfully!")

Trained segmentation models saved successfully!
